# rerank_condition_match — different-model topicality (SapBERT), the orthogonal lever

Cosine between the patient topic and the trial's topicality blob (conditions/title/summary), using
**SapBERT** (a biomedical entity bi-encoder) — a *different model* from Qwen (judges) and BioLinkBERT
(cross-encoders). The multi-view result showed the two LLM judges cap at r=0.67 because they share a model;
a different-model topicality signal should be more orthogonal. Cheap: embeddings + cosine over the whole pool.


## Setup (Colab — GPU makes the encode fast; SapBERT is small)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, topicality_blob, ndcg_at_k
POOL_TAG = 'nqs'   # 'R' for the original hybrid pool
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)
print('encoder:', cfg.topicality_encoder)

In [ ]:
SETS = ['trec21', 'kz', 'trec22']
OUT_PATH = cfg.feat_file('condition_match')
corpus_ids, corpus_fields = load_corpus(cfg); id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, SETS)
pool = json.load(open(cfg.pool_path()))
uniq = sorted({d for s in SETS for docs in pool[s].values() for d in docs if d in id2fields})
print(f'{len(uniq):,} unique pool docs to encode')


In [ ]:
# Encode the whole pool's topicality blobs + all topics with SapBERT (normalized -> cosine = dot).
enc = SentenceTransformer(cfg.topicality_encoder)
doc_emb = enc.encode([topicality_blob(id2fields[d], cfg) for d in uniq], normalize_embeddings=True,
                     batch_size=128, show_progress_bar=True).astype('float32')
didx = {d: i for i, d in enumerate(uniq)}
with open(OUT_PATH, 'w') as out:
    for s in SETS:
        tids = [t for t in pool[s] if t in sets[s]['topic2text']]
        q = enc.encode([sets[s]['topic2text'][t] for t in tids], normalize_embeddings=True).astype('float32')
        for t, qv in zip(tids, q):
            for d in pool[s][t]:
                if d in didx:
                    out.write(json.dumps({'source': s, 'topic_id': t, 'doc_id': d,
                                          'condition_match': float(qv @ doc_emb[didx[d]])}) + '\n')
print('wrote', OUT_PATH)


## Expanded condition_match: LLM-expanded query → SapBERT

The diagnostic showed buried rel=2 eligibles score cm ≈ −8 (displacers score +4). Flooring at 0
had zero effect — LambdaMART already treats all negative values the same, so the harm is that
displacers get *rewarded* for matching the condition name, not that buried eligibles are penalized.
The fix: expand the patient's topic with Qwen-generated synonyms/diagnoses before encoding,
so trials using different terminology for the same condition score higher.

Writes `condition_match_exp_{pool_tag}.jsonl` (separate from the original so A/B is possible).
To use it in `train_ensemble_full`, set `CM_PATH = cfg.feat_file('condition_match_exp')`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from ctmatch.experiments import llm_expand_query, resolve_ckpt

print('Loading Qwen for query expansion (GPU)...')
llm_tok = AutoTokenizer.from_pretrained(resolve_ckpt(cfg, cfg.llm_ckpt))
llm_model = AutoModelForCausalLM.from_pretrained(
    resolve_ckpt(cfg, cfg.llm_ckpt), torch_dtype=torch.bfloat16, device_map='auto')
llm_model.eval()

# Generate expansions for every unique (source, topic) pair across all sets.
expansions = {}
for s in SETS:
    for t, topic_text in sets[s]['topic2text'].items():
        if (s, t) not in expansions:
            exp = llm_expand_query(llm_model, llm_tok, topic_text, cfg)
            expansions[(s, t)] = exp

del llm_model; torch.cuda.empty_cache()
print(f'Generated {len(expansions)} topic expansions.')

# Spot-check a few.
for (s, t), exp in list(expansions.items())[:3]:
    print(f'\n[{s}/{t}] {sets[s]["topic2text"][t][:80]}')
    print(f'  -> {exp}')

In [ ]:
# Re-encode topics with the LLM-expanded query, then compute cosine against the same doc embeddings.
# doc_emb and didx are still in scope from the original encode cell above.
EXP_OUT_PATH = cfg.feat_file('condition_match_exp')
with open(EXP_OUT_PATH, 'w') as out:
    for s in SETS:
        tids = [t for t in pool[s] if t in sets[s]['topic2text']]
        expanded = [sets[s]['topic2text'][t] + '. ' + expansions.get((s, t), '') for t in tids]
        q_exp = enc.encode(expanded, normalize_embeddings=True,
                           batch_size=128, show_progress_bar=False).astype('float32')
        for t, qv in zip(tids, q_exp):
            for d in pool[s][t]:
                if d in didx:
                    out.write(json.dumps({'source': s, 'topic_id': t, 'doc_id': d,
                                          'condition_match': float(qv @ doc_emb[didx[d]])}) + '\n')
print('wrote', EXP_OUT_PATH)

In [ ]:
# Compare original vs expanded condition_match: standalone NDCG + score shift on buried_r2 docs.
# Load both files directly so this cell is independent of cell order.
cm = {}
for l in open(OUT_PATH):
    r = json.loads(l); cm[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']

cm_exp = {}
for l in open(EXP_OUT_PATH):
    r = json.loads(l); cm_exp[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']

# Per-split standalone NDCG (single-feature ranking).
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']; orig_vals, exp_vals = [], []
    for t, docs in pool[s].items():
        docs_ok = [d for d in docs if (s,t,d) in cm and (s,t,d) in cm_exp]
        orig_vals.append(ndcg_at_k(sorted(docs_ok, key=lambda d: -cm[(s,t,d)]),     rel[t]))
        exp_vals.append( ndcg_at_k(sorted(docs_ok, key=lambda d: -cm_exp[(s,t,d)]), rel[t]))
    rows.append({'split': s,
                 'cm_orig ndcg@10': round(float(np.mean(orig_vals)), 4),
                 'cm_exp  ndcg@10': round(float(np.mean(exp_vals)),  4),
                 'delta':           round(float(np.mean(exp_vals)) - float(np.mean(orig_vals)), 4)})
import pandas as pd; print(pd.DataFrame(rows).to_string(index=False))

# Score shift on the buried_r2 docs from the NQS diagnostic (if predictions exist).
nqs_pred_path = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs').feat_file('eval_predictions_ensemble')
if os.path.exists(nqs_pred_path):
    import json as _json
    buried_ids = {_json.loads(l)['doc_id']
                  for l in open(nqs_pred_path)
                  if _json.loads(l).get('label', 0) == 2 and _json.loads(l).get('rank', 999) > 10}
    buried_keys = [k for k in cm if k[2] in buried_ids and k in cm_exp]
    if buried_keys:
        orig_b = np.array([cm[k]     for k in buried_keys])
        exp_b  = np.array([cm_exp[k] for k in buried_keys])
        print(f'\nBuried_r2 score shift (n={len(buried_keys)}):')
        print(f'  cm_orig  mean={orig_b.mean():.3f}  pct>0={np.mean(orig_b>0)*100:.1f}%')
        print(f'  cm_exp   mean={exp_b.mean():.3f}  pct>0={np.mean(exp_b>0)*100:.1f}%')
        print(f'  mean delta = {(exp_b - orig_b).mean():+.3f}')
else:
    print('\n(no eval_predictions_ensemble_nqs.jsonl — skip buried_r2 score shift check)')

In [ ]:
# Standalone NDCG + correlations with the existing features (want LOW corr = orthogonal).
cm = {}
for l in open(OUT_PATH):
    r = json.loads(l); cm[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']; vals = []
    for t, docs in pool[s].items():
        docs = [d for d in docs if (s,t,d) in cm]
        vals.append(ndcg_at_k(sorted(docs, key=lambda d: cm[(s,t,d)], reverse=True), rel[t]))
    rows.append({'split': s, 'condition_match_ndcg@10': round(float(np.mean(vals)), 4)})
# correlations vs dense / clf_rel / llm_yesno / topicality on shared keys
def load_feat(name, key):
    d = {}
    for l in open(cfg.feat_file(name)):
        r = json.loads(l); d[(r['source'], r['topic_id'], r['doc_id'])] = r.get(key)
    return d
others = {'dense': load_feat('retrieval_feats', 'dense'),
          'llm_yesno': load_feat('llm_scores', 'llm_score')}
if os.path.exists(cfg.feat_file('topicality')): others['topicality'] = load_feat('topicality', 'topicality')
for name, f in others.items():
    ks = [k for k in cm if k in f and f[k] is not None]
    c = np.corrcoef([cm[k] for k in ks], [f[k] for k in ks])[0,1]
    print(f'corr(condition_match, {name}) = {c:.3f}')
pd.DataFrame(rows)


## Reading it
The win condition is **low correlation** with `dense`/`topicality` (and with the eligibility features) —
lower than the 0.67 the LLM topicality judge hit. If it's orthogonal, `train_ensemble_full` auto-includes
`condition_match_R.jsonl` and it should survive selection and add. If it's ~as correlated as the LLM judge,
a different-model encoder didn't help and we move to lever 2 (retrieval) or 3 (monoT5).
